In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from collections import Counter
from xgboost import XGBClassifier
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'xgboost'

In [3]:
!pip install xgboost


^C


  Using cached xgboost-3.0.0-py3-none-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ----

In [ ]:

# === Step 1: Load dataset ===
df = pd.read_csv('/kaggle/input/cleaned-diet/detailed_meals_macros_CLEANED.csv')

# === Step 2: Clean column names ===
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
print("Cleaned Columns:", df.columns.tolist())

# === Step 3: Encode categorical columns ===
for col in ['gender', 'activity_level', 'dietary_preference', 'disease']:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

In [ ]:
# === Step 4: Feature Engineering ===
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)
df['cal_per_kg'] = df['daily_calorie_target'] / df['weight']

total_macros = df['protein'] + df['carbohydrates'] + df['fat']
df['protein_pct'] = df['protein'] / total_macros
df['carbs_pct'] = df['carbohydrates'] / total_macros
df['fat_pct'] = df['fat'] / total_macros

# === Step 5: Define input features ===
features = ['ages', 'gender', 'height', 'weight', 'activity_level', 'dietary_preference',
            'disease', 'bmi', 'cal_per_kg', 'daily_calorie_target',
            'protein_pct', 'carbs_pct', 'fat_pct', 'fiber', 'sodium']

# === Step 6: Define target labels for each meal ===
targets = {
    'Breakfast': 'breakfast_suggestion',
    'Lunch': 'lunch_suggestion',
    'Dinner': 'dinner_suggestion',
    'Snack': 'snack_suggestion'
}

# === Step 7: Define base models ===
models = {
    'rf': RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced'),
    'gb': GradientBoostingClassifier(n_estimators=150, random_state=42),
    'xgb': XGBClassifier(n_estimators=150, learning_rate=0.05, use_label_encoder=False,
                        eval_metric='mlogloss', random_state=42)
}

# === Step 8: Create directory for saved models ===
os.makedirs("saved_models", exist_ok=True)

In [ ]:
# === Step 9: Train models for each meal ===
results = {}

for meal, target_col in targets.items():
    print(f"\n=== Training Model for {meal} Suggestion ===")

    X = df[features]
    y = df[target_col]

    # Encode target labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Filter rare classes (less than 6 samples)
    class_counts = Counter(y_encoded)
    valid_classes = [cls for cls, count in class_counts.items() if count >= 6]

    mask = np.isin(y_encoded, valid_classes)
    X = X[mask]
    y_encoded = y_encoded[mask]

    # Re-encode after filtering
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_encoded)

    # Balance using SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y_encoded)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_resampled, y_resampled, test_size=0.2, random_state=42)

    # Create soft voting ensemble model
    model = VotingClassifier(
        estimators=[(name, clf) for name, clf in models.items()],
        voting='soft'
    )

    # Train model
    model.fit(X_train, y_train)

    # Evaluation
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    top3_acc = top_k_accuracy_score(y_test, model.predict_proba(X_test), k=3)

    print(f"{meal} Model Trained ✅")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top-3 Accuracy: {top3_acc:.4f}")
    print(f"Final Classes Used: {len(np.unique(y_encoded))}")

    # Save model and label encoder
    model_path = f"saved_models/{meal.lower()}_model.pkl"
    encoder_path = f"saved_models/{meal.lower()}_label_encoder.pkl"

    joblib.dump(model, model_path)
    joblib.dump(le, encoder_path)

    print(f"Model saved to: {model_path}")
    print(f"LabelEncoder saved to: {encoder_path}")

    # Store results
    results[meal] = {
        'model': model,
        'encoder': le,
        'accuracy': acc,
        'top3_accuracy': top3_acc
    }

# === All models trained and saved successfully ===
print("\n🎉 All Meal Models Trained & Saved Successfully!")


Cleaned Columns: ['ages', 'gender', 'height', 'weight', 'activity_level', 'dietary_preference', 'daily_calorie_target', 'protein', 'sugar', 'sodium', 'calories', 'carbohydrates', 'fiber', 'fat', 'breakfast_suggestion', 'breakfast_calories', 'breakfast_protein', 'breakfast_carbohydrates', 'breakfast_fats', 'lunch_suggestion', 'lunch_calories', 'lunch_protein', 'lunch_carbohydrates', 'dinner_suggestion', 'dinner_calories', 'dinner_protein.1', 'dinner_carbohydrates.1', 'dinner_fats', 'snack_suggestion', 'snacks_calories', 'snacks_protein', 'snacks_carbohydrates', 'snacks_fats', 'disease', 'lunch_fats']

=== Training Model for Breakfast Suggestion ===
Breakfast Model Trained ✅
Accuracy: 0.7491
Top-3 Accuracy: 0.9107
Final Classes Used: 28
Model saved to: saved_models/breakfast_model.pkl
LabelEncoder saved to: saved_models/breakfast_label_encoder.pkl

=== Training Model for Lunch Suggestion ===
Lunch Model Trained ✅
Accuracy: 0.6714
Top-3 Accuracy: 0.8693
Final Classes Used: 24
Model saved 